In [1]:
!pip install -q groq python-dotenv pydantic ollama

In [24]:
import os
import json
import ollama
import time
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, ValidationError
from pathlib import Path

In [3]:
load_dotenv()
client = Groq()

In [20]:
MODELO = "openai/gpt-oss-20b"

# 1. ESQUEMA PYDANTIC PARA LA SALIDA FINAL
class Metadatos(BaseModel):
    categoria: str
    nivel_urgencia: str

class Analisis(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: str

class AccionRespuesta(BaseModel):
    codigo_accion_sistema: str
    mensaje_al_cliente: str

class SalidaMaestra(BaseModel):
    metadatos_clasificacion: Metadatos
    analisis_interno: Analisis
    accion_y_respuesta: AccionRespuesta

# 2. FUNCIÓN AUXILIAR PARA LLAMAR A GROQ
def llamar_groq_json(system_prompt: str, input_data: dict, max_tokens: int) -> dict:
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": f"{system_prompt}\n\nDebes responder ÚNICAMENTE con un objeto JSON válido que cumpla con el OUTPUT esperado."
            },
            {
                "role": "user",
                "content": json.dumps(input_data)
            }
        ],
        model=MODELO,
        response_format={"type": "json_object"},
        temperature=0.1,
        max_tokens=max_tokens
    )
    return json.loads(chat_completion.choices[0].message.content)

# 3. FUNCIÓN PRINCIPAL ACTUALIZADA
def procesar_mensaje(mensaje_usuario: str) -> dict:
    try:
        prompt_urgencia = open("prompt_urgencia.md", "r", encoding="utf-8").read()
        prompt_categoria = open("prompt_categoria.md", "r", encoding="utf-8").read()
        prompt_accion = open("prompt_accion.md", "r", encoding="utf-8").read()
        prompt_texto = open("prompt_texto.md", "r", encoding="utf-8").read()
        prompt_maestro = open("prompt_maestro.md", "r", encoding="utf-8").read()
    except FileNotFoundError as e:
        raise RuntimeError(f"Error al leer los archivos de prompt: {e}")

    # PASO 1: Urgencia
    out_urgencia = llamar_groq_json(prompt_urgencia, {"mensaje_usuario": mensaje_usuario}, max_tokens=100)
    nivel_urgencia = out_urgencia.get("nivel_urgencia", "Desconocido")
    time.sleep(15)

    # PASO 2: Categoría
    out_categoria = llamar_groq_json(prompt_categoria, {"mensaje_usuario": mensaje_usuario}, max_tokens=100)
    categoria = out_categoria.get("categoria", "Desconocida")
    time.sleep(15)

    # PASO 3: Acción
    out_accion = llamar_groq_json(
        prompt_accion,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria},
        max_tokens=400
    )
    cadena_de_pensamiento = out_accion.get("cadena_de_pensamiento", "")
    politica_aplicada = out_accion.get("politica_aplicada", "")
    codigo_accion_sistema = out_accion.get("codigo_accion_sistema", "")
    time.sleep(15)

    # PASO 4: Texto
    out_texto = llamar_groq_json(
        prompt_texto,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria, "politica_aplicada": politica_aplicada},
        max_tokens=400
    )
    mensaje_al_cliente = out_texto.get("mensaje_al_cliente", "")
    time.sleep(15)

    # PASO 5: Maestro
    out_maestro = llamar_groq_json(
        prompt_maestro,
        {
            "categoria": categoria,
            "nivel_urgencia": nivel_urgencia,
            "cadena_de_pensamiento": cadena_de_pensamiento,
            "politica_aplicada": politica_aplicada,
            "codigo_accion_sistema": codigo_accion_sistema,
            "mensaje_al_cliente": mensaje_al_cliente
        },
        max_tokens=800
    )

    salida_validada = SalidaMaestra(**out_maestro)
    return salida_validada.model_dump()

In [21]:
# EJEMPLO DE USO 1
if __name__ == "__main__":
    mensaje = "Hola, compré una cafetera hace dos meses y ayer dejó de prender. Además me cobraron dos veces el envío en la tarjeta. Quiero mi plata ya o los demando."
    try:
        resultado = procesar_mensaje(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

Error procesando el mensaje: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}


In [25]:
# 1. Definición de Esquemas Pydantic

class MetadatosClasificacion(BaseModel):
    categoria: str
    nivel_urgencia: str

class AnalisisInterno(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: str

class AccionYRespuesta(BaseModel):
    codigo_accion_sistema: str
    mensaje_al_cliente: str

class SalidaMaestro(BaseModel):
    metadatos_clasificacion: MetadatosClasificacion
    analisis_interno: AnalisisInterno
    accion_y_respuesta: AccionYRespuesta

# 2. Funciones Auxiliares

def cargar_prompt(nombre_archivo: str) -> str:
    """Lee el contenido de un archivo de prompt .md"""
    ruta = Path(nombre_archivo)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo {nombre_archivo}")
    return ruta.read_text(encoding='utf-8')

def llamar_llm(system_prompt: str, input_data: dict, modelo: str = "qwen3:8b") -> dict:

    mensaje_usuario = f"{system_prompt}\n\n### INPUT ###\n{json.dumps(input_data, ensure_ascii=False, indent=2)}"

    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {
                'role': 'system',
                'content': 'Sos un asistente experto. Debes responder estrictamente en formato JSON válido basándote en el esquema solicitado.'
            },
            {
                'role': 'user',
                'content': mensaje_usuario
            }
        ],
        format='json',
        options={'temperature': 0.1}
    )

    try:
        return json.loads(respuesta['message']['content'])
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Error al decodificar la respuesta JSON del LLM: {respuesta['message']['content']}") from e

# 3. Función Principal del Pipeline

def procesar_mensaje(mensaje_usuario: str) -> dict:
    modelo_llm = "qwen3:8b"

    # --- PASO 1: Urgencia ---
    prompt_urgencia = cargar_prompt("prompt_urgencia.md")
    res_urgencia = llamar_llm(prompt_urgencia, {"mensaje_usuario": mensaje_usuario}, modelo_llm)
    nivel_urgencia = res_urgencia.get("nivel_urgencia", "")

    # --- PASO 2: Categoría ---
    prompt_categoria = cargar_prompt("prompt_categoria.md")
    res_categoria = llamar_llm(prompt_categoria, {"mensaje_usuario": mensaje_usuario}, modelo_llm)
    categoria = res_categoria.get("categoria", "")

    # --- PASO 3: Acción ---
    prompt_accion = cargar_prompt("prompt_accion.md")
    input_accion = {
        "mensaje_usuario": mensaje_usuario,
        "nivel_urgencia": nivel_urgencia,
        "categoria": categoria
    }
    res_accion = llamar_llm(prompt_accion, input_accion, modelo_llm)

    # --- PASO 4: Texto (Mensaje al Cliente) ---
    prompt_texto = cargar_prompt("prompt_texto.md")
    input_texto = {
        "mensaje_usuario": mensaje_usuario,
        "nivel_urgencia": nivel_urgencia,
        "categoria": categoria,
        "politica_aplicada": res_accion.get("politica_aplicada", "")
    }
    res_texto = llamar_llm(prompt_texto, input_texto, modelo_llm)

    # --- PASO 5: Maestro (Estructuración Final) ---
    prompt_maestro = cargar_prompt("prompt_maestro.md")
    input_maestro = {
        "categoria": categoria,
        "nivel_urgencia": nivel_urgencia,
        "cadena_de_pensamiento": res_accion.get("cadena_de_pensamiento", ""),
        "politica_aplicada": res_accion.get("politica_aplicada", ""),
        "codigo_accion_sistema": res_accion.get("codigo_accion_sistema", ""),
        "mensaje_al_cliente": res_texto.get("mensaje_al_cliente", "")
    }
    res_maestro = llamar_llm(prompt_maestro, input_maestro, modelo_llm)

    # --- VALIDACIÓN FINAL CON PYDANTIC ---
    try:
        resultado_validado = SalidaMaestro(**res_maestro)
        return resultado_validado.model_dump() # Retorna el diccionario validado
    except ValidationError as e:
        print("Error de validación en la estructura final devuelta por el Prompt Maestro:")
        print(e)
        # Fallback de seguridad en caso de que el LLM devuelva un JSON mal estructurado
        return res_maestro


In [ ]:
# Ejemplo de uso 1
if __name__ == "__main__":
    mensaje_prueba = "Hola, compré una cafetera hace dos meses y ayer dejó de prender. Además me cobraron dos veces el envío en la tarjeta. Quiero mi plata ya o los demando."
    resultado = procesar_mensaje(mensaje_prueba)

    print("\n=== RESULTADO FINAL ESTRUCTURADO ===")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))

In [ ]:
# Ejemplo de uso 2
if __name__ == "__main__":
    mensaje_prueba = "El estado de mi pedido dice 'entregado' hace 3 días, pero yo estaba de viaje y mis cámaras muestran que nadie vino. Para colmo, en el resumen de la tarjeta me aparece un recargo del 15% que no reconozco. Ayuda urgente."
    resultado = procesar_mensaje(mensaje_prueba)

    print("\n=== RESULTADO FINAL ESTRUCTURADO ===")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))

In [ ]:
# ==========================================
# Ejemplo de uso 3
# ==========================================
if __name__ == "__main__":
    mensaje_prueba = "Compré 4 sillas y una mesa. Llegaron hoy, pero 2 sillas tienen las patas astilladas y la mesa no trajo los tornillos para armarla. Tengo una cena mañana y necesito esto resuelto hoy mismo."
    resultado = procesar_mensaje(mensaje_prueba)

    print("\n=== RESULTADO FINAL ESTRUCTURADO ===")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))

In [ ]:
# ==========================================
# Ejemplo de uso 4
# ==========================================
if __name__ == "__main__":
    mensaje_prueba = "Estoy intentando descargar mi factura A desde la plataforma pero el PDF da error. Además, la caja del mouse que compré vino aplastada, aunque el mouse funciona bien. Lo que más me importa es la factura para el contador hoy."
    resultado = procesar_mensaje(mensaje_prueba)

    print("\n=== RESULTADO FINAL ESTRUCTURADO ===")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))

In [ ]:
# ==========================================
# Ejemplo de uso 5
# ==========================================
if __name__ == "__main__":
    mensaje_prueba = "El repartidor me acaba de llamar pidiendo que le pague un 'extra por zona peligrosa' en efectivo. Me negué y se fue con mi paquete (una notebook de alta gama). Ya pagué el envío premium con la tarjeta. ¡Exijo que me lo traigan ya y sancionen al repartidor!"
    resultado = procesar_mensaje(mensaje_prueba)

    print("\n=== RESULTADO FINAL ESTRUCTURADO ===")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))